In [3]:
import pandas as pd
import re
from datetime import datetime, timedelta
import jieba
from snownlp import SnowNLP

F:\Users\lenovo\anaconda\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# 第一部分：导入库与定义通用辅助函数

In [29]:
# 辅助函数1：解析微博时间字符串
def parse_weibo_time(time_str, crawl_date=None):
    """
    支持格式：
      - '02月27日 12:42'          (默认年份为爬取年份)
      - '2025年11月11日 18:50'
      - '今天13:12'               (替换为crawl_date当天)
      - '04月13日 16:43'
    """
    if pd.isna(time_str):
        return pd.NaT
    s = str(time_str).strip()
    if crawl_date is None:
        crawl_date = datetime.now()
    # 处理“今天”
    if s.startswith('今天'):
        today_str = crawl_date.strftime('%Y年%m月%d日')
        s = s.replace('今天', today_str)
    # 如果只有月日，补年份
    if re.match(r'\d{2}月\d{2}日', s):
        s = f"{crawl_date.year}年{s}"
    # 多种格式尝试
    patterns = [
        r'(\d{4})年(\d{1,2})月(\d{1,2})日\s*(\d{1,2}):(\d{2})',
        r'(\d{4})-(\d{1,2})-(\d{1,2})\s+(\d{1,2}):(\d{2})',
        r'(\d{4})/(\d{1,2})/(\d{1,2})\s+(\d{1,2}):(\d{2})'
    ]
    for pat in patterns:
        m = re.search(pat, s)
        if m:
            y, mo, d, h, minute = map(int, m.groups())
            return datetime(y, mo, d, h, minute)
    return pd.NaT

# 辅助函数2：将“1.2万”或“125”转换为整数
def parse_count(count_str):
    if pd.isna(count_str):
        return 0
    if isinstance(count_str, (int, float)):
        return int(count_str)
    s = str(count_str).strip()
    if '万' in s:
        num = float(s.replace('万', ''))
        return int(num * 10000)
    else:
        num = re.sub(r'[^\d]', '', s)
        return int(num) if num else 0

# 辅助函数3：统计文本中包含的关键词个数
def count_keywords(text, kw_list):
    if pd.isna(text):
        return 0
    text = str(text)
    return sum(1 for kw in kw_list if kw in text)

# ========== 修改后的关键词词典（聚焦真人声线模仿） ==========
core_ai_words = [
    'AI翻唱', 'AI cover', '声音克隆', '声线模仿', '声线替换', '用AI唱',
    'AI孙燕姿', 'AI那英', 'AI周杰伦', 'AI王菲', 'AI林俊杰', 'AI声线'
]

exclude_words = [
    'AI绘画', 'AI写作', 'AI换脸', 'AI视频', 'AI设计',
    '真人翻唱', '翻唱比赛', '翻唱专辑',
    '招聘', '广告', '贷款', '抽奖', '兼职',
    '大头针', '林栖', 'AI大头针', '虚拟歌手', '洛天依', '初音未来'
]

companionship_words = [
    '陪伴','治愈','温暖','孤独',
    '安慰','不孤单','精神寄托',
    '解压','放松', '怀念','想念',
    '回忆','青春',
    '泪目','破防',
    '感动','听哭',
    '回来了','再听到',
    '爷青回'
]
fidelity_words = [
    '像','太像',
    '好像', '还原','神还原','逼真', '一模一样', '以假乱真','音色', '声音','声线','本人','真假难辨','听不出来'
]
concern_words = [
    '侵权','版权', '法律','未授权','赔偿', '违法', '维权', '官司', '声音权', '肖像权', '授权', '同意', '伦理',
    '冒犯', '尊重', '消费逝者'
]

print("库导入和辅助函数定义完成。")

库导入和辅助函数定义完成。


# 第二部分：加载原始数据

In [30]:
# 读取两个Excel文件
posts_df = pd.read_excel('Weibo_Posts.xlsx')
reviews_df = pd.read_excel('Weibo_Reviews.xlsx')

print(f"原始帖子数：{len(posts_df)}")
print(f"原始评论行数：{len(reviews_df)}")
print("帖子表列名：", posts_df.columns.tolist())

原始帖子数：969
原始评论行数：4052
帖子表列名： ['关键词', '页面网址', '博主昵称', '发布时间', '详情链接', '博文内容', '图片链接', '视频链接', '转发数', '评论数', '点赞数', '当前时间', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']


# 第三部分：清洗帖子表（Weibo_Posts）

## 3.1 重命名关键列

In [31]:
# 删除所有 Unnamed 开头的空列
posts_df = posts_df.loc[:, ~posts_df.columns.str.contains('^Unnamed', na=False)]
print("删除无用列后的列名：", posts_df.columns.tolist())
#重命名关键列
posts_df.rename(columns={
    '关键词': 'keyword',
    '页面网址': 'search_url',
    '博主昵称': 'author_nick',
    '发布时间': 'publish_time_raw',
    '详情链接': 'post_url',
    '博文内容': 'content',
    '图片链接': 'image_links',
    '视频链接': 'video_link',
    '转发数': 'reposts_raw',
    '评论数': 'comments_count_raw',
    '点赞数': 'likes_raw',
    '当前时间': 'crawl_time'
}, inplace=True)


删除无用列后的列名： ['关键词', '页面网址', '博主昵称', '发布时间', '详情链接', '博文内容', '图片链接', '视频链接', '转发数', '评论数', '点赞数', '当前时间']


## 3.2 删除完全重复的帖子（基于post_url）

In [32]:
posts_df.drop_duplicates(subset=['post_url'], keep='first', inplace=True)
print(f"去重后帖子数：{len(posts_df)}")

去重后帖子数：912


## 3.3 处理缺失值

In [33]:
posts_df['content'] = posts_df['content'].fillna('')
posts_df['image_links'] = posts_df['image_links'].fillna('')
posts_df['video_link'] = posts_df['video_link'].fillna('')

## 3.4 解析发布时间

In [34]:
crawl_time = datetime.now()  # 可用文件中的当前时间列，这里简单用系统时间
posts_df['publish_datetime'] = posts_df['publish_time_raw'].apply(lambda x: parse_weibo_time(x, crawl_time))
posts_df.dropna(subset=['publish_datetime'], inplace=True)
posts_df['publish_date'] = posts_df['publish_datetime'].dt.date
posts_df['publish_hour'] = posts_df['publish_datetime'].dt.hour
print(f"成功解析时间的帖子数：{len(posts_df)}")

成功解析时间的帖子数：910


## 3.5 转换互动数据

In [35]:
posts_df['reposts'] = posts_df['reposts_raw'].apply(parse_count)
posts_df['comments_count_show'] = posts_df['comments_count_raw'].apply(parse_count)
posts_df['likes'] = posts_df['likes_raw'].apply(parse_count)
posts_df.drop(['reposts_raw', 'comments_count_raw', 'likes_raw'], axis=1, inplace=True)

## 3.6 内容清洗（去换行、多余空格）

In [36]:
posts_df['content_clean'] = posts_df['content'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip())

## 3.7 相关性过滤（使用修改后的词典）

In [37]:
def is_relevant(text):
    text_lower = str(text).lower()
    has_core = any(kw.lower() in text_lower for kw in core_ai_words)
    if has_core:
        return True
    has_exclude = any(kw.lower() in text_lower for kw in exclude_words)
    return not has_exclude

posts_df['is_relevant'] = posts_df['content_clean'].apply(is_relevant)
relevant_posts = posts_df[posts_df['is_relevant'] == True].copy()
print(f"相关帖子数：{len(relevant_posts)}，剔除无关帖子：{len(posts_df)-len(relevant_posts)}")

相关帖子数：899，剔除无关帖子：11


## 3.8 计算情感得分和关键词得分

In [38]:
def get_sentiment(text):
    if not text or len(text) < 2:
        return 0.5
    try:
        return SnowNLP(text).sentiments
    except:
        return 0.5

relevant_posts['sentiment'] = relevant_posts['content_clean'].apply(get_sentiment)
relevant_posts['content_length'] = relevant_posts['content_clean'].apply(len)
relevant_posts['has_video'] = relevant_posts['video_link'].apply(lambda x: 1 if x and str(x).strip() else 0)
relevant_posts['has_image'] = relevant_posts['image_links'].apply(lambda x: 1 if x and str(x).strip() else 0)

# 关键词得分（原始计数）
relevant_posts['companionship_raw'] = relevant_posts['content_clean'].apply(lambda x: count_keywords(x, companionship_words))
relevant_posts['fidelity_raw'] = relevant_posts['content_clean'].apply(lambda x: count_keywords(x, fidelity_words))
relevant_posts['concern_raw'] = relevant_posts['content_clean'].apply(lambda x: count_keywords(x, concern_words))

# 标准化（Min-Max 到 0-1）
for col in ['companionship_raw', 'fidelity_raw', 'concern_raw']:
    max_val = relevant_posts[col].max()
    if max_val > 0:
        relevant_posts[col+'_norm'] = relevant_posts[col] / max_val
    else:
        relevant_posts[col+'_norm'] = 0

# 临时接受意愿代理（后续会结合评论）
relevant_posts['discussion_attitude'] = relevant_posts['sentiment']

## 3.9 输出清洗后的帖子表（暂存）

In [39]:
posts_clean = relevant_posts.copy()
posts_clean.to_excel('Weibo_Posts_cleaned_temp.xlsx', index=False)
print("帖子表清洗临时文件已保存。")

帖子表清洗临时文件已保存。


# 第四部分：清洗评论表（Weibo_Reviews）

## 4.1 重命名列

In [40]:
# 4.1 重命名列（先删除原有的错误post_url列，再重命名正确的列）
# 删除错误的 post_url 列（如果存在）
if 'post_url' in reviews_df.columns:
    reviews_df.drop(columns=['post_url'], inplace=True)

# 重命名正确的列
reviews_df.rename(columns={
    '博文网址': 'post_url',           # 真正的帖子链接
    '博文内容': 'post_content',
    '博文图片链接': 'post_images',
    '博文视频链接': 'post_video',
    '转发数': 'post_reposts',
    '评论数': 'post_comments',
    '点赞数': 'post_likes',
    '评论人': 'commenter_nick',
    '评论人链接': 'commenter_url',
    '评论内容': 'comment_content',
    '评论时间': 'comment_time_raw',
    '评论获赞': 'comment_likes',
    '二级评论人': 'sub_commenter_nick',
    '二级评论内容': 'sub_comment_content',
    '二级评论时间': 'sub_comment_time_raw',
    '二级评论人链接': 'sub_commenter_url'
}, inplace=True)

# 验证
print("评论表 post_url 列前3条（正确链接）：")
print(reviews_df['post_url'].head(3))

评论表 post_url 列前3条（正确链接）：
0    https://weibo.com/6266195272/PFJcPudRU?refer_f...
1    https://weibo.com/6091463374/R0idl0m13?refer_f...
2    https://weibo.com/6091463374/R0idl0m13?refer_f...
Name: post_url, dtype: object


## 4.2 处理一级评论和二级评论（合并为统一评论表）

In [41]:
# 4.2 处理一级评论和二级评论（合并为统一评论表）

# 一级评论（没有二级内容）
primary = reviews_df[reviews_df['sub_comment_content'].isna()].copy()
primary['is_sub'] = 0
primary['comment_clean'] = primary['comment_content']
primary['comment_time'] = primary['comment_time_raw']
primary['commenter'] = primary['commenter_nick']
primary['likes'] = primary['comment_likes'].fillna(0)

# 二级评论（有二级内容）
sub = reviews_df[reviews_df['sub_comment_content'].notna()].copy()
if not sub.empty:
    sub['is_sub'] = 1
    sub['comment_clean'] = sub['sub_comment_content']
    sub['comment_time'] = sub['sub_comment_time_raw']
    sub['commenter'] = sub['sub_commenter_nick']
    sub['likes'] = 0
else:
    sub = pd.DataFrame(columns=primary.columns)

# 合并
all_comments = pd.concat([primary, sub], ignore_index=True)
print(f"一级评论数：{len(primary)}")
print(f"二级评论数：{len(sub)}")
print(f"合并后评论总行数：{len(all_comments)}")

一级评论数：1843
二级评论数：2209
合并后评论总行数：4052


## 4.3 删除重复评论（基于评论人+内容+时间）

In [42]:
all_comments.drop_duplicates(subset=['commenter', 'comment_clean', 'comment_time'], keep='first', inplace=True)
print(f"去重后评论数：{len(all_comments)}")

去重后评论数：3758


## 4.4 解析评论时间

In [43]:
all_comments['comment_datetime'] = all_comments['comment_time'].apply(lambda x: parse_weibo_time(x, crawl_time))
all_comments.dropna(subset=['comment_datetime'], inplace=True)
all_comments['comment_date'] = all_comments['comment_datetime'].dt.date

## 4.5 清洗评论内容

In [44]:
all_comments['comment_clean'] = all_comments['comment_clean'].apply(lambda x: re.sub(r'\s+', ' ', str(x)).strip() if pd.notna(x) else '')
all_comments = all_comments[all_comments['comment_clean'].str.len() > 0]

## 4.6 评论情感得分和关键词得分

In [45]:
all_comments['comment_sentiment'] = all_comments['comment_clean'].apply(get_sentiment)
all_comments['companionship_raw'] = all_comments['comment_clean'].apply(lambda x: count_keywords(x, companionship_words))
all_comments['fidelity_raw'] = all_comments['comment_clean'].apply(lambda x: count_keywords(x, fidelity_words))
all_comments['concern_raw'] = all_comments['comment_clean'].apply(lambda x: count_keywords(x, concern_words))

for col in ['companionship_raw', 'fidelity_raw', 'concern_raw']:
    max_val = all_comments[col].max()
    if max_val > 0:
        all_comments[col+'_norm'] = all_comments[col] / max_val
    else:
        all_comments[col+'_norm'] = 0

# 第五部分：验证每条帖子的评论爬取完整性

In [46]:
import re

def extract_weibo_id(url):
    if pd.isna(url):
        return ''
    match = re.search(r'(?:weibo\.com)?/(\d+)/([A-Za-z0-9]+)', url)
    if match:
        user_id, post_id = match.groups()
        return f"{user_id}/{post_id}"
    return ''

posts_clean['post_url_base'] = posts_clean['post_url'].apply(extract_weibo_id)
all_comments['post_url_base'] = all_comments['post_url'].apply(extract_weibo_id)

counts_dict = all_comments.groupby('post_url_base').size().to_dict()
posts_clean['actual_comment_count'] = posts_clean['post_url_base'].map(counts_dict).fillna(0).astype(int)

print("actual_comment_count 是否全为 0？", (posts_clean['actual_comment_count'] == 0).all())
print("有评论的帖子数：", (posts_clean['actual_comment_count'] > 0).sum())

def compute_coverage(row):
    show = row['comments_count_show']
    actual = row['actual_comment_count']
    if show == 0:
        return 1.0 if actual > 0 else 1.0
    else:
        return actual / show

posts_clean['coverage'] = posts_clean.apply(compute_coverage, axis=1)
posts_clean['comments_reliable'] = (posts_clean['coverage'] > 0.5) | (posts_clean['actual_comment_count'] > 20)

print("\n=== 修正后的评论爬取完整性报告 ===")
print(f"总相关帖子数：{len(posts_clean)}")
print(f"有评论的帖子数：{(posts_clean['actual_comment_count'] > 0).sum()}")
print(f"覆盖率均值：{posts_clean['coverage'].mean():.2f}")
print(f"覆盖率中位数：{posts_clean['coverage'].median():.2f}")
print(f"评论可靠帖子数：{posts_clean['comments_reliable'].sum()}")

actual_comment_count 是否全为 0？ False
有评论的帖子数： 494

=== 修正后的评论爬取完整性报告 ===
总相关帖子数：899
有评论的帖子数：494
覆盖率均值：0.88
覆盖率中位数：1.00
评论可靠帖子数：810


In [47]:
# 假设 posts_clean 已有 columns: comments_count_show, actual_comment_count
def classify_comment_status(row):
    show = row['comments_count_show']
    actual = row['actual_comment_count']
    if show == 0 and actual == 0:
        return '真无评论'
    elif show > 0 and actual == 0:
        return '漏爬（有评论但未爬到）'
    elif actual >= show * 0.8:  # 覆盖率 ≥ 80%
        return '完整爬取'
    else:
        return '部分爬取'

posts_clean['comment_status'] = posts_clean.apply(classify_comment_status, axis=1)

# 统计各类别数量
print(posts_clean['comment_status'].value_counts())

posts_clean['comments_usable'] = posts_clean['comment_status'].isin(['真无评论', '完整爬取', '部分爬取'])

comment_status
真无评论           389
完整爬取           312
部分爬取           182
漏爬（有评论但未爬到）     16
Name: count, dtype: int64


In [48]:
print("=== 精细化数据质量报告 ===")
print(f"总相关帖子：{len(posts_clean)}")
print(f"实际爬到至少一条评论的帖子：{(posts_clean['actual_comment_count'] > 0).sum()}")
print(f"真无评论（显示0且爬取0）：{(posts_clean['comment_status'] == '真无评论').sum()}")
print(f"漏爬（显示>0但爬取0）：{(posts_clean['comment_status'] == '漏爬（有评论但未爬到）').sum()}")
print(f"可用于评论分析的帖子（真无评论+完整/部分爬取）：{posts_clean['comments_usable'].sum()}")

=== 精细化数据质量报告 ===
总相关帖子：899
实际爬到至少一条评论的帖子：494
真无评论（显示0且爬取0）：389
漏爬（显示>0但爬取0）：16
可用于评论分析的帖子（真无评论+完整/部分爬取）：883


In [49]:
if not all_comments.empty:
    comment_agg = all_comments.groupby('post_url').agg({
        'comment_sentiment': 'mean',
        'concern_raw_norm': 'mean',
        'companionship_raw_norm': 'mean',
        'fidelity_raw_norm': 'mean',
        'likes': 'sum',
        'comment_clean': 'count'
    }).rename(columns={
        'comment_sentiment': 'avg_comment_sentiment',
        'concern_raw_norm': 'avg_comment_concern',
        'companionship_raw_norm': 'avg_comment_companionship',
        'fidelity_raw_norm': 'avg_comment_fidelity',
        'likes': 'total_comment_likes',
        'comment_clean': 'num_comments_crawled'
    })
    posts_clean = posts_clean.merge(comment_agg, left_on='post_url', right_index=True, how='left')
    fill_cols = ['avg_comment_sentiment', 'avg_comment_concern', 'avg_comment_companionship',
                 'avg_comment_fidelity', 'total_comment_likes', 'num_comments_crawled']
    for col in fill_cols:
        if col in posts_clean.columns:
            posts_clean[col] = posts_clean[col].fillna(0)
    # 更新接受意愿代理
    posts_clean['discussion_attitude'] = (posts_clean['sentiment'] + posts_clean['avg_comment_sentiment']) / 2
else:
    print("警告：没有评论数据，无法聚合评论特征。")

In [50]:
# 保存帖子表
posts_clean.to_excel('Weibo_Posts_cleaned.xlsx', index=False)
print("帖子表最终保存：Weibo_Posts_cleaned.xlsx")

# 保存评论表（仅保留属于相关帖子的评论）
relevant_urls = set(posts_clean['post_url'].unique())
comments_final = all_comments[all_comments['post_url'].isin(relevant_urls)]
comments_final.to_excel('Weibo_Reviews_cleaned.xlsx', index=False)
print(f"评论表最终保存：Weibo_Reviews_cleaned.xlsx，共{len(comments_final)}条评论")

# 删除临时文件
import os
if os.path.exists('Weibo_Posts_cleaned_temp.xlsx'):
    os.remove('Weibo_Posts_cleaned_temp.xlsx')
print("清洗完成。")

帖子表最终保存：Weibo_Posts_cleaned.xlsx
评论表最终保存：Weibo_Reviews_cleaned.xlsx，共3607条评论
清洗完成。
